# Домашнее задание 2

## Анализ данных о пассажирах «Титаника»


# 1. Загрузка датасета

Считаем файл `train.csv` с помощью `pandas.read_csv()` и выведем первые строки, чтобы убедиться, что данные загрузились корректно.

In [1]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Также посмотрим размер таблицы: количество строк и столбцов.

In [2]:
print("Количество строк:", df.shape[0])
print("Количество столбцов:", df.shape[1])

Количество строк: 891
Количество столбцов: 12


# 2. Основная информация о датасете

Сначала выведем общую информацию о столбцах и типах данных и посчитаем количество пропущенных значений в каждом столбце.

In [4]:
df.info()

missing_values = df.isna().sum()
missing_values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Теперь выведем описательную статистику по числовым столбцам: количество наблюдений, среднее значение, стандартное отклонение, минимум, максимум и квартили.

In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# 3. Процент выживаемости в каждом классе

Столбец `Survived` содержит:
- `1` — пассажир выжил;
- `0` — пассажир не выжил.

Поэтому среднее значение `Survived` внутри каждого класса равно доле выживших. Умножим её на 100, чтобы получить процент.

In [6]:
survival_by_class = (
    df.groupby("Pclass")["Survived"]
      .mean()
      .mul(100)
      .round(2)
)

survival_by_class

Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64

Из результатов видно, что процент выживаемости был самым высоким среди пассажиров 1 класса и самым низким среди пассажиров 3 класса.

# 4. Самое популярное мужское и женское имя

В столбце `Name` имя записано вместе с фамилией и обращением.
Для поиска именно личного имени создадим функцию `extract_first_name`.

У замужних женщин настоящее имя часто указано в скобках после имени мужа.

Поэтому для женщин при наличии скобок будем брать первое имя из скобок. В остальных случаях — первое слово после обращения (`Mr.`, `Miss.`, `Master.` и т.д.).

In [9]:
def extract_first_name(full_name, sex):
    if sex == "female" and "(" in full_name:
        name = full_name.split("(")[1]
        name = name.replace(")", "")
        first_name = name.split()[0]
    else:
        name = full_name.split(".")[1]
        first_name = name.strip().split()[0]

    return first_name


df["FirstName"] = df.apply(
    lambda row: extract_first_name(row["Name"], row["Sex"]),
    axis=1
)

df[["Name", "Sex", "FirstName"]]

,Name,Sex,FirstName
0,"Braund, Mr. Owen Harris",male,Owen
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,Florence
2,"Heikkinen, Miss. Laina",female,Laina
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,Lily
4,"Allen, Mr. William Henry",male,William
...,...,...,...
886,"Montvila, Rev. Juozas",male,Juozas
887,"Graham, Miss. Margaret Edith",female,Margaret
888,"Johnston, Miss. Catherine Helen ""Carrie""",female,Catherine
889,"Behr, Mr. Karl Howell",male,Karl


Теперь посчитаем частоту каждого имени отдельно для мужчин и женщин.

Если несколько имён встречаются одинаковое максимальное число раз, выведем их все.

In [10]:
male_counts = df.loc[df["Sex"] == "male", "FirstName"].value_counts()
female_counts = df.loc[df["Sex"] == "female", "FirstName"].value_counts()

most_common_male = male_counts[male_counts == male_counts.max()]
most_common_female = female_counts[female_counts == female_counts.max()]

print("Самое популярное мужское имя:")
print(most_common_male)

print("\nСамое популярное женское имя / имена:")
print(most_common_female)

Самое популярное мужское имя:
FirstName
William    35
Name: count, dtype: int64

Самое популярное женское имя / имена:
FirstName
Anna    15
Name: count, dtype: int64


В датасете самое частое мужское имя — **William**.  
Среди женщин максимальная частота одинаковая у **Anna**

# 5. Самые популярные мужские и женские имена в каждом классе

Для каждого класса отдельно найдём максимальную частоту имени среди мужчин и среди женщин.  
При совпадении частот также сохраним все имена-лидеры.

In [11]:
def most_popular_names(data: pd.DataFrame, passenger_class: int, sex: str):
    names = data.loc[
        (data["Pclass"] == passenger_class) & (data["Sex"] == sex),
        "FirstName"
    ]

    counts = names.value_counts()
    max_count = counts.max()

    return counts[counts == max_count]


for passenger_class in sorted(df["Pclass"].unique()):
    print(f"Класс {passenger_class}")

    print("  Мужчины:")
    print(most_popular_names(df, passenger_class, "male"))

    print("  Женщины:")
    print(most_popular_names(df, passenger_class, "female"))

    print()

Класс 1
  Мужчины:
FirstName
William    11
Name: count, dtype: int64
  Женщины:
FirstName
Elizabeth    5
Margaret     5
Name: count, dtype: int64

Класс 2
  Мужчины:
FirstName
William    9
Name: count, dtype: int64
  Женщины:
FirstName
Elizabeth    5
Name: count, dtype: int64

Класс 3
  Мужчины:
FirstName
William    15
Name: count, dtype: int64
  Женщины:
FirstName
Anna    9
Name: count, dtype: int64




- **1 класс:** William среди мужчин; Elizabeth и Margaret среди женщин.
- **2 класс:** William среди мужчин; Elizabeth среди женщин.
- **3 класс:** William среди мужчин; Anna среди женщин.

# 6. Пассажиры старше 44 лет

Отфильтруем строки по условию `Age > 44`.  
Для компактного вывода покажем первые 10 подходящих пассажиров.

In [13]:
passengers_over_44 = df[df["Age"] > 44]

print("Количество пассажиров старше 44 лет:", len(passengers_over_44))
passengers_over_44

Количество пассажиров старше 44 лет: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S,Peter
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S,Margaret
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S,Sallie
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S,Victor


# 7. Мужчины младше 44 лет

Используем сразу два условия:
- возраст меньше 44 лет;
- пол — мужской.

Условия объединяются оператором `&`. Для вывода покажем первые 10 строк.

In [14]:
men_under_44 = df[
    (df["Age"] < 44) &
    (df["Sex"] == "male")
]

print("Количество мужчин младше 44 лет:", len(men_under_44))
men_under_44

Количество мужчин младше 44 лет: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S,Frederick
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S,Juozas
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C,Karl


# 8. Количество n-местных кабин

1. считаем, сколько пассажиров приходится на каждое указанное значение `Cabin`;
2. считаем, сколько таких кабин было занято 2, 3, 4 и более пассажирами;



In [15]:
passengers_per_cabin = df["Cabin"].dropna().value_counts()

n_person_cabins = (
    passengers_per_cabin
    .value_counts()
    .sort_index()
)

n_person_cabins = n_person_cabins[n_person_cabins.index >= 2]
n_person_cabins.index.name = "Количество пассажиров в кабине"
n_person_cabins.name = "Количество кабин"

n_person_cabins

Количество пассажиров в кабине
2    38
3     5
4     3
Name: Количество кабин, dtype: int64

По данным таблицы:
- 38 значений кабин встречаются у 2 пассажиров;
- 5 — у 3 пассажиров;
- 3 — у 4 пассажиров.

# 9. Пассажиры без родственников на борту

В датасете:
- `SibSp` — количество братьев, сестёр, супругов на борту;
- `Parch` — количество родителей и детей на борту.

Если оба значения равны нулю, у пассажира нет указанных родственников на борту.

In [16]:
passengers_without_relatives = df[
    (df["SibSp"] == 0) &
    (df["Parch"] == 0)
]

print(
    "Количество пассажиров без родственников на борту:",
    len(passengers_without_relatives)
)

Количество пассажиров без родственников на борту: 537
